In [ ]:
import pandas as pd
import numpy as np
import re

SHEETS = ["Y1S1","Y1S2","Y2S1","Y2S2","Y3S1","Y3S2","Y4S1","Y4S2"]
SHEET_TO_IDX = {s: i+1 for i, s in enumerate(SHEETS)}

def parse_level_type(module_code: str):
    if pd.isna(module_code):
        return (np.nan, np.nan)
    s = str(module_code).strip()
    m = re.search(r"-(\d)(\d)", s)   
    if not m:
        return (np.nan, np.nan)
    return (int(m.group(1)), int(m.group(2)))

def wavg(marks, weights):
    marks = pd.to_numeric(marks, errors="coerce")
    w = pd.to_numeric(weights, errors="coerce").fillna(0)
    mask = marks.notna() & (w > 0)
    if mask.sum() == 0:
        return np.nan
    return float((marks[mask] * w[mask]).sum() / w[mask].sum())

def build_checkpoint_features(df_sheet: pd.DataFrame, checkpoint_sheet: str) -> pd.DataFrame:
    df = df_sheet.copy()

    # Fix header issues once
    df.columns = df.columns.astype(str).str.strip()

    required = ["REGNO", "MODULE_CODE", "PASS_FAIL", "MC", "MARKS"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{checkpoint_sheet} missing columns: {missing}")

    df["REGNO"] = df["REGNO"].astype(str).str.strip()

    # Parse module level/type
    level_type = df["MODULE_CODE"].apply(parse_level_type)
    df["module_level"] = [x[0] for x in level_type]
    df["module_type"]  = [x[1] for x in level_type]

    # Numeric conversions
    df["MC"] = pd.to_numeric(df["MC"], errors="coerce").fillna(0)
    df["MARKS"] = pd.to_numeric(df["MARKS"], errors="coerce")

    # PASS/FAIL only 
    pf = df["PASS_FAIL"].astype(str).str.upper().str.strip()
    df["_is_pass"] = (pf == "PASS").astype(int)
    df["_is_fail"] = (pf == "FAIL").astype(int)

    rows = []
    for regno, g in df.groupby("REGNO", dropna=False):
        module_count = int(g["MODULE_CODE"].notna().sum())
        credits_attempted = float(g["MC"].sum())

        rows.append({
            "REGNO": regno,
            "checkpoint_idx": SHEET_TO_IDX[checkpoint_sheet],
            "checkpoint_sheet": checkpoint_sheet,
            "credits_attempted": credits_attempted,
            "module_count": module_count,
            "wavg_mark": wavg(g["MARKS"], g["MC"]),
            "avg_mark": float(g["MARKS"].mean()) if g["MARKS"].notna().any() else np.nan,
            "mark_min": float(g["MARKS"].min())  if g["MARKS"].notna().any() else np.nan,
            "mark_max": float(g["MARKS"].max())  if g["MARKS"].notna().any() else np.nan,
            "mark_std": float(g["MARKS"].std(ddof=0)) if g["MARKS"].notna().any() else np.nan,
            "fail_count": int(g["_is_fail"].sum()),
            "pass_count": int(g["_is_pass"].sum()),

            # credits + wavg by level (L1-L4)
            **{f"credits_L{L}": float(g.loc[g["module_level"]==L, "MC"].sum()) for L in [1,2,3,4]},
            **{f"wavg_mark_L{L}": wavg(g.loc[g["module_level"]==L, "MARKS"],
                                      g.loc[g["module_level"]==L, "MC"]) for L in [1,2,3,4]},

            # credits + wavg by type (T1-T5)
            **{f"credits_T{T}": float(g.loc[g["module_type"]==T, "MC"].sum()) for T in [1,2,3,4,5]},
            **{f"wavg_mark_T{T}": wavg(g.loc[g["module_type"]==T, "MARKS"],
                                      g.loc[g["module_type"]==T, "MC"]) for T in [1,2,3,4,5]},
        })

    return pd.DataFrame(rows)

In [ ]:
import pandas as pd
import numpy as np

TRANSCRIPT_PATH = r"C:\Users\User\Desktop\Final Year Project\Data\Transcript.xlsx"
CGPA_PATH       = r"C:\Users\User\Desktop\Final Year Project\Data\Transcript_cgpa.xlsx"

SHEETS = ["Y1S1","Y1S2","Y2S1","Y2S2","Y3S1","Y3S2","Y4S1","Y4S2"]


all_features = []
for sh in SHEETS:
    df_sh = pd.read_excel(TRANSCRIPT_PATH, sheet_name=sh)
    feat = build_checkpoint_features(df_sh, sh)
    all_features.append(feat)

features_df = pd.concat(all_features, ignore_index=True)


bins   = [0.0, 2.0, 3.0, 3.5, 4.0, 5.0]
labels = ["High Risk","Risk","Moderate","Safe","Very Safe"]

cgpa_y4s2 = pd.read_excel(CGPA_PATH, sheet_name="Y4S2")
cgpa_y4s2.columns = cgpa_y4s2.columns.astype(str).str.strip()
cgpa_y4s2["REGNO"] = cgpa_y4s2["REGNO"].astype(str).str.strip()
cgpa_y4s2["GPA"] = pd.to_numeric(cgpa_y4s2["GPA"], errors="coerce")

cgpa_y4s2["RISK_BAND"] = pd.cut(
    cgpa_y4s2["GPA"],
    bins=bins,
    labels=labels,
    include_lowest=True,
    right=True
).astype("string")

risk_df = cgpa_y4s2[["REGNO","RISK_BAND"]].drop_duplicates("REGNO")

final_df = features_df.merge(risk_df, on="REGNO", how="left")

final_columns = [
    "REGNO","checkpoint_idx","checkpoint_sheet","credits_attempted","module_count",
    "wavg_mark","avg_mark","mark_min","mark_max","mark_std","fail_count","pass_count",
    "credits_L1","credits_L2","credits_L3","credits_L4",
    "wavg_mark_L1","wavg_mark_L2","wavg_mark_L3","wavg_mark_L4",
    "credits_T1","credits_T2","credits_T3","credits_T4","credits_T5",
    "wavg_mark_T1","wavg_mark_T2","wavg_mark_T3","wavg_mark_T4","wavg_mark_T5",
    "RISK_BAND"
]
final_df = final_df[final_columns]

print("Rows:", len(final_df))
print("Unique students:", final_df["REGNO"].nunique())
print(final_df.head(3))

# ---- save ----
OUT_XLSX = r"C:\Users\User\Desktop\Final Year Project\Data\FeatureDataset.xlsx"
OUT_CSV  = r"C:\Users\User\Desktop\Final Year Project\Data\FeatureDataset.csv"

final_df.to_excel(OUT_XLSX, index=False)
final_df.to_csv(OUT_CSV, index=False)

print("Saved to:", OUT_XLSX)

Rows: 6242
Unique students: 849
    REGNO  checkpoint_idx checkpoint_sheet  credits_attempted  module_count  \
0  100001               1             Y1S1               16.0             4   
1  100002               1             Y1S1               16.0             4   
2  100003               1             Y1S1               20.0             5   

   wavg_mark  avg_mark  mark_min  mark_max   mark_std  ...  credits_T2  \
0      64.45     64.45      51.2      73.0   8.065513  ...         4.0   
1      64.55     64.55      54.4      75.2   7.455703  ...         4.0   
2      60.78     60.78      42.5      73.5  10.100178  ...         4.0   

   credits_T3  credits_T4  credits_T5  wavg_mark_T1  wavg_mark_T2  \
0         4.0         0.0         8.0           NaN          67.3   
1         4.0         0.0         8.0           NaN          62.6   
2         4.0         4.0         8.0           NaN          62.1   

   wavg_mark_T3  wavg_mark_T4  wavg_mark_T5  RISK_BAND  
0          51.2     

In [ ]:
import pandas as pd
import numpy as np

TRANSCRIPT_PATH = r"C:\Users\User\Desktop\Final Year Project\Data\Transcript.xlsx"
CGPA_PATH       = r"C:\Users\User\Desktop\Final Year Project\Data\Transcript_cgpa.xlsx"

SHEETS = ["Y1S1","Y1S2","Y2S1","Y2S2","Y3S1","Y3S2","Y4S1","Y4S2"]

bins   = [0.0, 2.0, 3.0, 3.5, 4.0, 5.0]
labels = ["High Risk","Risk","Moderate","Safe","Very Safe"]

# risk band table from Transcript_cgpa.xlsx Y4S2 GPA) ---
cgpa_y4s2 = pd.read_excel(CGPA_PATH, sheet_name="Y4S2")
cgpa_y4s2.columns = cgpa_y4s2.columns.astype(str).str.strip()
cgpa_y4s2["REGNO"] = cgpa_y4s2["REGNO"].astype(str).str.strip()
cgpa_y4s2["GPA"] = pd.to_numeric(cgpa_y4s2["GPA"], errors="coerce")

cgpa_y4s2["RISK_BAND"] = pd.cut(
    cgpa_y4s2["GPA"],
    bins=bins,
    labels=labels,
    include_lowest=True,
    right=True
).astype("string")

risk_df = cgpa_y4s2[["REGNO","RISK_BAND"]].drop_duplicates("REGNO")

# build per-sheet 
OUT_XLSX = r"C:\Users\User\Desktop\Final Year Project\Data\FeatureDataset.xlsx"

all_frames = []

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    for sh in SHEETS:
        df_sh = pd.read_excel(TRANSCRIPT_PATH, sheet_name=sh)
        feat = build_checkpoint_features(df_sh, sh)

        feat = feat.merge(risk_df, on="REGNO", how="left")

        feat = feat[
            ["REGNO","checkpoint_idx","checkpoint_sheet","credits_attempted","module_count",
             "wavg_mark","avg_mark","mark_min","mark_max","mark_std","fail_count","pass_count",
             "credits_L1","credits_L2","credits_L3","credits_L4",
             "wavg_mark_L1","wavg_mark_L2","wavg_mark_L3","wavg_mark_L4",
             "credits_T1","credits_T2","credits_T3","credits_T4","credits_T5",
             "wavg_mark_T1","wavg_mark_T2","wavg_mark_T3","wavg_mark_T4","wavg_mark_T5",
             "RISK_BAND"]
        ]

        feat.to_excel(writer, sheet_name=sh, index=False)
        all_frames.append(feat)

        
    all_df = pd.concat(all_frames, ignore_index=True)
    all_df.to_excel(writer, sheet_name="ALL", index=False)

print("Saved multi-sheet dataset to:", OUT_XLSX)

Saved multi-sheet dataset to: C:\Users\User\Desktop\Final Year Project\Data\FeatureDataset.xlsx


In [ ]:
import pandas as pd
import numpy as np

DATA_PATH = r"C:\Users\User\Desktop\Final Year Project\Data\FeatureDataset.xlsx"

df = pd.read_excel(DATA_PATH, sheet_name="ALL")

df.columns = df.columns.astype(str).str.strip()
df["REGNO"] = df["REGNO"].astype(str).str.strip()
df["checkpoint_sheet"] = df["checkpoint_sheet"].astype(str).str.strip()

In [6]:
SHEETS_ORDER = ["Y1S1","Y1S2","Y2S1","Y2S2","Y3S1","Y3S2","Y4S1","Y4S2"]

def cumulative_df(upto_idx):
    keep = set(SHEETS_ORDER[:upto_idx])
    return df[df["checkpoint_sheet"].isin(keep)].copy()

In [7]:
from sklearn.model_selection import GroupShuffleSplit

def group_train_test_split(data, group_col="REGNO", test_size=0.2, random_state=42):
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    X = data
    groups = data[group_col].values
    train_idx, test_idx = next(gss.split(X, groups=groups))
    return data.iloc[train_idx].copy(), data.iloc[test_idx].copy()

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score, accuracy_score

from pycaret.classification import ClassificationExperiment

DATA_PATH = r"C:\Users\User\Desktop\Final Year Project\Data\FeatureDataset.xlsx"

df = pd.read_excel(DATA_PATH, sheet_name="ALL")

df.columns = df.columns.astype(str).str.strip()
df["REGNO"] = df["REGNO"].astype(str).str.strip()
df["checkpoint_sheet"] = df["checkpoint_sheet"].astype(str).str.strip()

SHEETS_ORDER = ["Y1S1","Y1S2","Y2S1","Y2S2","Y3S1","Y3S2","Y4S1","Y4S2"]

def cumulative_df(upto_idx):
    keep = set(SHEETS_ORDER[:upto_idx])
    return df[df["checkpoint_sheet"].isin(keep)].copy()

def group_train_test_split(data, group_col="REGNO", test_size=0.2, random_state=42):
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    idx = np.arange(len(data))
    train_idx, test_idx = next(gss.split(idx, groups=data[group_col].values))
    return data.iloc[train_idx].copy(), data.iloc[test_idx].copy()

results = []

for k in range(1, 9):
    data_k = cumulative_df(k)
    data_k = data_k[data_k["RISK_BAND"].notna()].copy()

    train_k, test_k = group_train_test_split(data_k, group_col="REGNO", test_size=0.2, random_state=42)

    exp = ClassificationExperiment()
    exp.setup(
    data=train_k,
    target="RISK_BAND",
    session_id=42,
    fold=5,
    fold_strategy="groupkfold",
    fold_groups="REGNO",                
    ignore_features=["REGNO","checkpoint_sheet"],
    normalize=True,
    verbose=False
)

    best = exp.compare_models(sort="F1")

    preds = exp.predict_model(best, data=test_k)  

    y_true = test_k["RISK_BAND"].astype(str).values
    y_pred = preds["prediction_label"].astype(str).values

    f1_macro = f1_score(y_true, y_pred, average="macro")
    acc = accuracy_score(y_true, y_pred)

    results.append({
        "checkpoint_upto": SHEETS_ORDER[k-1],
        "n_rows": len(data_k),
        "n_students": data_k["REGNO"].nunique(),
        "best_model": str(best),
        "test_f1_macro": float(f1_macro),
        "test_accuracy": float(acc),
    })

results_df = pd.DataFrame(results)

# F1_macro first, then accuracy
ranked = results_df.sort_values(["test_f1_macro", "test_accuracy"], ascending=False)

print("=== RESULTS (in time order) ===")
print(results_df)

print("\n=== RANKED (best first: Macro-F1 then Accuracy) ===")
print(ranked)

# Earliest checkpoint macro-F1
best_f1 = ranked.iloc[0]["test_f1_macro"]
tol = 0.02  
earliest_good = results_df[results_df["test_f1_macro"] >= best_f1 - tol].iloc[0]

print("\nBest Macro-F1:", best_f1)
print("Earliest checkpoint within tol =", tol, ":", earliest_good["checkpoint_upto"])

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
gbc,Gradient Boosting Classifier,0.4000,0.0000,0.4000,0.4160,0.3967,0.2270,0.2308,0.2360
et,Extra Trees Classifier,0.3581,0.6546,0.3581,0.3719,0.3529,0.1717,0.1749,0.0660
rf,Random Forest Classifier,0.3613,0.6533,0.3613,0.3646,0.3483,0.1726,0.1773,0.1080
knn,K Neighbors Classifier,0.3548,0.6037,0.3548,0.3610,0.3444,0.1686,0.1728,0.0440
dt,Decision Tree Classifier,0.3355,0.5724,0.3355,0.3513,0.3362,0.1489,0.1507,0.0240
lightgbm,Light Gradient Boosting Machine,0.3387,0.6592,0.3387,0.3483,0.3341,0.1536,0.1562,0.2180
svm,SVM - Linear Kernel,0.3258,0.0000,0.3258,0.3466,0.3208,0.1385,0.1416,0.0240
ridge,Ridge Classifier,0.3129,0.0000,0.3129,0.3281,0.3028,0.1192,0.1240,0.0160
ada,Ada Boost Classifier,0.3129,0.0000,0.3129,0.3045,0.3025,0.1213,0.1229,0.0540
lda,Linear Discriminant Analysis,0.3032,0.0000,0.3032,0.3168,0.2936,0.1048,0.1089,0.0160


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Gradient Boosting Classifier,0.2411,0.5556,0.2411,0.2615,0.2376,0.0276,0.0283


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
gbc,Gradient Boosting Classifier,0.3307,0.0000,0.3307,0.3629,0.3332,0.1428,0.1452,0.4180
rf,Random Forest Classifier,0.3252,0.6487,0.3252,0.3521,0.3215,0.1342,0.1372,0.1000
et,Extra Trees Classifier,0.3213,0.6434,0.3213,0.3440,0.3208,0.1280,0.1299,0.0900
lda,Linear Discriminant Analysis,0.3266,0.0000,0.3266,0.3429,0.3188,0.1364,0.1396,0.0200
lr,Logistic Regression,0.3199,0.0000,0.3199,0.3427,0.3175,0.1289,0.1312,0.0240
ridge,Ridge Classifier,0.3199,0.0000,0.3199,0.3290,0.3049,0.1244,0.1282,0.0160
lightgbm,Light Gradient Boosting Machine,0.3063,0.6307,0.3063,0.3154,0.3031,0.1107,0.1120,0.6660
knn,K Neighbors Classifier,0.3009,0.5922,0.3009,0.3365,0.3003,0.1050,0.1077,0.0200
dt,Decision Tree Classifier,0.2915,0.5482,0.2915,0.2974,0.2899,0.0977,0.0986,0.0200
ada,Ada Boost Classifier,0.2848,0.0000,0.2848,0.2947,0.2813,0.0905,0.0916,0.0520


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Gradient Boosting Classifier,0.2809,0.6001,0.2809,0.3053,0.2770,0.0893,0.0916


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
et,Extra Trees Classifier,0.3655,0.6584,0.3655,0.3854,0.3636,0.1845,0.1877,0.1440
lightgbm,Light Gradient Boosting Machine,0.3604,0.6517,0.3604,0.3813,0.3603,0.1795,0.1818,1.1920
gbc,Gradient Boosting Classifier,0.3544,0.0000,0.3544,0.3697,0.3525,0.1672,0.1694,0.8560
rf,Random Forest Classifier,0.3518,0.6601,0.3518,0.3720,0.3488,0.1655,0.1682,0.1920
lda,Linear Discriminant Analysis,0.3536,0.0000,0.3536,0.3618,0.3476,0.1692,0.1719,0.0280
lr,Logistic Regression,0.3518,0.0000,0.3518,0.3600,0.3434,0.1683,0.1717,0.0360
ridge,Ridge Classifier,0.3433,0.0000,0.3433,0.3537,0.3339,0.1526,0.1560,0.0260
knn,K Neighbors Classifier,0.3331,0.6236,0.3331,0.3404,0.3243,0.1416,0.1443,0.0280
dt,Decision Tree Classifier,0.3177,0.5648,0.3177,0.3231,0.3170,0.1333,0.1340,0.0280
ada,Ada Boost Classifier,0.2989,0.0000,0.2989,0.3040,0.2921,0.1051,0.1071,0.0840


...

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,0.3402,0.6349,0.3402,0.3591,0.3428,0.1532,0.1547,0.2780
lr,Logistic Regression,0.3461,0.0000,0.3461,0.3760,0.3380,0.1529,0.1577,0.0440
gbc,Gradient Boosting Classifier,0.3329,0.0000,0.3329,0.3598,0.3325,0.1373,0.1402,1.3460
et,Extra Trees Classifier,0.3322,0.6309,0.3322,0.3457,0.3321,0.1445,0.1460,0.1920
lightgbm,Light Gradient Boosting Machine,0.3290,0.6424,0.3290,0.3486,0.3311,0.1378,0.1393,0.9660
lda,Linear Discriminant Analysis,0.3388,0.0000,0.3388,0.3653,0.3290,0.1430,0.1480,0.0240
qda,Quadratic Discriminant Analysis,0.3231,0.0000,0.3231,0.3565,0.3244,0.1443,0.1491,0.0220
ridge,Ridge Classifier,0.3346,0.0000,0.3346,0.3585,0.3204,0.1349,0.1403,0.0260
ada,Ada Boost Classifier,0.3217,0.0000,0.3217,0.3366,0.3162,0.1233,0.1257,0.0980
knn,K Neighbors Classifier,0.3140,0.6132,0.3140,0.3200,0.3081,0.1203,0.1227,0.0360


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Random Forest Classifier,0.3149,0.6169,0.3149,0.3440,0.3173,0.1311,0.1335


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lightgbm,Light Gradient Boosting Machine,0.4034,0.7088,0.4034,0.4200,0.4040,0.2309,0.2330,0.9460
et,Extra Trees Classifier,0.3958,0.6879,0.3958,0.4070,0.3953,0.2231,0.2249,0.2240
rf,Random Forest Classifier,0.3955,0.6943,0.3955,0.4090,0.3950,0.2218,0.2236,0.3060
gbc,Gradient Boosting Classifier,0.3940,0.0000,0.3940,0.4186,0.3908,0.2143,0.2188,1.5000
lr,Logistic Regression,0.3791,0.0000,0.3791,0.3984,0.3668,0.1930,0.1990,0.0440
lda,Linear Discriminant Analysis,0.3785,0.0000,0.3785,0.3969,0.3665,0.1922,0.1981,0.0240
knn,K Neighbors Classifier,0.3569,0.6504,0.3569,0.3648,0.3538,0.1715,0.1734,0.0400
ada,Ada Boost Classifier,0.3544,0.0000,0.3544,0.3652,0.3475,0.1659,0.1690,0.1060
ridge,Ridge Classifier,0.3627,0.0000,0.3627,0.3860,0.3454,0.1690,0.1762,0.0220
dt,Decision Tree Classifier,0.3316,0.5838,0.3316,0.3372,0.3318,0.1461,0.1467,0.0320


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Light Gradient Boosting Machine,0.3607,0.6876,0.3607,0.3956,0.3614,0.1890,0.1932


=== RESULTS (in time order) ===
  checkpoint_upto  n_rows  n_students  \
0            Y1S1     556         556   
1            Y1S2    1326         771   
2            Y2S1    2096         771   
3            Y2S2    2864         771   
4            Y3S1    3628         771   
5            Y3S2    4383         771   
6            Y4S1    5124         771   
7            Y4S2    5877         771   

                                          best_model  test_f1_macro  \
0  GradientBoostingClassifier(ccp_alpha=0.0, crit...       0.235596   
1  GradientBoostingClassifier(ccp_alpha=0.0, crit...       0.278206   
2  ExtraTreesClassifier(bootstrap=False, ccp_alph...       0.343018   
3  RandomForestClassifier(bootstrap=True, ccp_alp...       0.314754   
4  RandomForestClassifier(bootstrap=True, ccp_alp...       0.299891   
5  ExtraTreesClassifier(bootstrap=False, ccp_alph...       0.300597   
6  RandomForestClassifier(bootstrap=True, ccp_alp...       0.322761   
7  LGBMClassifier(boosting_typ

In [13]:
stage1 = results_df[results_df["checkpoint_upto"]=="Y1S1"][["test_accuracy","test_f1_macro"]]
stage2_all = results_df[results_df["checkpoint_upto"]=="Y4S2"][["test_accuracy","test_f1_macro"]]

print("Stage 1 (Y1S1):")
print(stage1.to_string(index=False))

print("\nStage 2 (All up to Y4S2):")
print(stage2_all.to_string(index=False))

Stage 1 (Y1S1):
 test_accuracy  test_f1_macro
      0.241071       0.235596

Stage 2 (All up to Y4S2):
 test_accuracy  test_f1_macro
      0.360711       0.368741


In [ ]:
import pandas as pd
import numpy as np

FEATURE_PATH = r"C:\Users\User\Desktop\Final Year Project\Data\FeatureDataset.xlsx"
CGPA_PATH    = r"C:\Users\User\Desktop\Final Year Project\Data\Transcript_cgpa.xlsx"
OUT_PATH     = r"C:\Users\User\Desktop\Final Year Project\Data\FeatureDataset_RiskProgress.xlsx"

SHEETS = ["Y1S1","Y1S2","Y2S1","Y2S2","Y3S1","Y3S2","Y4S1","Y4S2"]

bins   = [0.0, 2.0, 3.0, 3.5, 4.0, 5.0]
labels = ["High Risk","Risk","Moderate","Safe","Very Safe"]

def make_risk_df(sheet_name):
    t = pd.read_excel(CGPA_PATH, sheet_name=sheet_name)
    t.columns = t.columns.astype(str).str.strip()
    t["REGNO"] = t["REGNO"].astype(str).str.strip()
    t["GPA"] = pd.to_numeric(t["GPA"], errors="coerce")
    t["RISK_BAND"] = pd.cut(t["GPA"], bins=bins, labels=labels, include_lowest=True, right=True).astype("string")
    return t[["REGNO","RISK_BAND"]].drop_duplicates("REGNO")

#target risk (Y4S2)
risk_y4s2 = make_risk_df("Y4S2").rename(columns={"RISK_BAND":"RISK_BAND_Y4S2"})

with pd.ExcelWriter(OUT_PATH, engine="openpyxl") as writer:
    for sh in SHEETS:
        df = pd.read_excel(FEATURE_PATH, sheet_name=sh)
        df.columns = df.columns.astype(str).str.strip()
        df["REGNO"] = df["REGNO"].astype(str).str.strip()

        risk_curr = make_risk_df(sh)   

        if "RISK_BAND" in df.columns:
            df = df.drop(columns=["RISK_BAND"])

        df = df.merge(risk_curr, on="REGNO", how="left")


        df = df.merge(risk_y4s2, on="REGNO", how="left")

        df.to_excel(writer, sheet_name=sh, index=False)

print("Saved:", OUT_PATH)

Saved: C:\Users\User\Desktop\Final Year Project\Data\FeatureDataset_RiskProgress.xlsx


In [18]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score, accuracy_score

from pycaret.classification import ClassificationExperiment

DATA_PATH = r"C:\Users\User\Desktop\Final Year Project\Data\FeatureDataset_RiskProgress.xlsx"

SHEETS_ORDER = ["Y1S1","Y1S2","Y2S1","Y2S2","Y3S1","Y3S2","Y4S1","Y4S2"]
TARGET = "RISK_BAND_Y4S2"

def load_concat(upto_idx):
    frames = []
    for sh in SHEETS_ORDER[:upto_idx]:
        d = pd.read_excel(DATA_PATH, sheet_name=sh)
        d.columns = d.columns.astype(str).str.strip()
        d["REGNO"] = d["REGNO"].astype(str).str.strip()
        frames.append(d)
    return pd.concat(frames, ignore_index=True)

def group_train_test_split(data, group_col="REGNO", test_size=0.2, random_state=42):
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    idx = np.arange(len(data))
    train_idx, test_idx = next(gss.split(idx, groups=data[group_col].values))
    return data.iloc[train_idx].copy(), data.iloc[test_idx].copy()

results = []

for k in range(1, 9):
    data_k = load_concat(k)
    data_k = data_k[data_k[TARGET].notna()].copy()

    train_k, test_k = group_train_test_split(data_k, group_col="REGNO", test_size=0.2, random_state=42)

    exp = ClassificationExperiment()
    exp.setup(
        data=train_k.drop(columns=["REGNO"]),
        target=TARGET,
        session_id=42,
        fold=5,
        normalize=True,
        verbose=False
    )

    best = exp.compare_models(sort="F1")
    preds = exp.predict_model(best, data=test_k.drop(columns=["REGNO"]))

    y_true = test_k[TARGET].astype(str).values
    y_pred = preds["prediction_label"].astype(str).values

    f1_macro = f1_score(y_true, y_pred, average="macro")
    acc = accuracy_score(y_true, y_pred)

    results.append({
        "checkpoint_upto": SHEETS_ORDER[k-1],
        "n_rows": len(data_k),
        "n_students": data_k["REGNO"].nunique(),
        "best_model": str(best),
        "test_f1_macro": float(f1_macro),
        "test_accuracy": float(acc),
    })

results_df = pd.DataFrame(results)

print("=== Results (time order) ===")
print(results_df)

print("\n=== Ranked by Macro-F1 then Accuracy ===")
print(results_df.sort_values(["test_f1_macro","test_accuracy"], ascending=False))

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
gbc,Gradient Boosting Classifier,0.3903,0.0000,0.3903,0.4068,0.3901,0.2149,0.2173,0.2620
rf,Random Forest Classifier,0.3871,0.6716,0.3871,0.3905,0.3805,0.2128,0.2153,0.0860
et,Extra Trees Classifier,0.3839,0.6599,0.3839,0.3959,0.3797,0.2110,0.2139,0.0840
lightgbm,Light Gradient Boosting Machine,0.3484,0.6428,0.3484,0.3529,0.3433,0.1612,0.1633,0.2880
knn,K Neighbors Classifier,0.3548,0.6303,0.3548,0.3524,0.3402,0.1697,0.1745,0.6460
ada,Ada Boost Classifier,0.3290,0.0000,0.3290,0.3314,0.3243,0.1432,0.1443,0.0580
ridge,Ridge Classifier,0.3323,0.0000,0.3323,0.3438,0.3234,0.1416,0.1448,0.0280
lda,Linear Discriminant Analysis,0.3290,0.0000,0.3290,0.3384,0.3224,0.1381,0.1404,0.0300
lr,Logistic Regression,0.3290,0.0000,0.3290,0.3326,0.3220,0.1376,0.1396,1.3660
dt,Decision Tree Classifier,0.3000,0.5525,0.3000,0.3044,0.2984,0.1106,0.1113,0.0280


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Gradient Boosting Classifier,0.2768,0.5617,0.2768,0.2878,0.2695,0.0765,0.0784


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
et,Extra Trees Classifier,0.3576,0.6534,0.3576,0.3600,0.3534,0.1744,0.1759,0.1180
rf,Random Forest Classifier,0.3522,0.6584,0.3522,0.3657,0.3460,0.1624,0.1655,0.1040
gbc,Gradient Boosting Classifier,0.3414,0.0000,0.3414,0.3430,0.3367,0.1524,0.1539,0.7340
lightgbm,Light Gradient Boosting Machine,0.3334,0.6354,0.3334,0.3378,0.3284,0.1409,0.1426,0.6260
lda,Linear Discriminant Analysis,0.3347,0.0000,0.3347,0.3234,0.3189,0.1439,0.1468,0.0380
lr,Logistic Regression,0.3307,0.0000,0.3307,0.3181,0.3171,0.1375,0.1397,0.0580
ridge,Ridge Classifier,0.3361,0.0000,0.3361,0.3118,0.3116,0.1432,0.1474,0.0280
dt,Decision Tree Classifier,0.3117,0.5631,0.3117,0.3114,0.3093,0.1273,0.1277,0.0300
ada,Ada Boost Classifier,0.3117,0.0000,0.3117,0.3098,0.3053,0.1228,0.1239,0.1180
knn,K Neighbors Classifier,0.3064,0.6042,0.3064,0.3057,0.2927,0.1066,0.1093,0.0320


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Extra Trees Classifier,0.3146,0.5845,0.3146,0.3332,0.3007,0.1337,0.1383


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,0.3698,0.6706,0.3698,0.3824,0.3679,0.1855,0.1873,0.1420
gbc,Gradient Boosting Classifier,0.3664,0.0000,0.3664,0.3720,0.3638,0.1844,0.1856,0.5980
lightgbm,Light Gradient Boosting Machine,0.3519,0.6534,0.3519,0.3600,0.3511,0.1642,0.1651,1.0740
lda,Linear Discriminant Analysis,0.3578,0.0000,0.3578,0.3583,0.3503,0.1739,0.1760,0.0320
et,Extra Trees Classifier,0.3484,0.6499,0.3484,0.3526,0.3460,0.1621,0.1632,0.1140
lr,Logistic Regression,0.3518,0.0000,0.3518,0.3520,0.3432,0.1633,0.1655,0.0500
ridge,Ridge Classifier,0.3535,0.0000,0.3535,0.3575,0.3416,0.1652,0.1684,0.0300
knn,K Neighbors Classifier,0.3382,0.6100,0.3382,0.3304,0.3252,0.1480,0.1505,0.0340
ada,Ada Boost Classifier,0.3322,0.0000,0.3322,0.3273,0.3232,0.1450,0.1466,0.0780
dt,Decision Tree Classifier,0.3023,0.5552,0.3023,0.3010,0.3001,0.1125,0.1128,0.0320


...

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,0.3517,0.6432,0.3517,0.3603,0.3518,0.1658,0.1666,0.2500
gbc,Gradient Boosting Classifier,0.3521,0.0000,0.3521,0.3678,0.3485,0.1590,0.1613,1.3520
et,Extra Trees Classifier,0.3472,0.6317,0.3472,0.3522,0.3470,0.1610,0.1616,0.2000
lr,Logistic Regression,0.3510,0.0000,0.3510,0.3713,0.3455,0.1562,0.1587,0.0680
lda,Linear Discriminant Analysis,0.3500,0.0000,0.3500,0.3681,0.3448,0.1541,0.1565,0.0420
ridge,Ridge Classifier,0.3500,0.0000,0.3500,0.3689,0.3400,0.1515,0.1550,0.0400
lightgbm,Light Gradient Boosting Machine,0.3343,0.6454,0.3343,0.3437,0.3346,0.1417,0.1425,0.8940
ada,Ada Boost Classifier,0.3367,0.0000,0.3367,0.3425,0.3313,0.1456,0.1474,0.1260
knn,K Neighbors Classifier,0.3133,0.6008,0.3133,0.3139,0.3067,0.1146,0.1161,0.0560
nb,Naive Bayes,0.3144,0.6233,0.3144,0.3176,0.3033,0.1271,0.1299,0.0400


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Random Forest Classifier,0.3256,0.6236,0.3256,0.3542,0.3270,0.1437,0.1464


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,0.4338,0.7193,0.4338,0.4399,0.4329,0.2714,0.2726,0.2740
et,Extra Trees Classifier,0.4329,0.7102,0.4329,0.4366,0.4316,0.2710,0.2721,0.2180
lightgbm,Light Gradient Boosting Machine,0.4296,0.7318,0.4296,0.4376,0.4291,0.2650,0.2663,0.9560
gbc,Gradient Boosting Classifier,0.4323,0.0000,0.4323,0.4475,0.4286,0.2634,0.2667,1.6060
lda,Linear Discriminant Analysis,0.4208,0.0000,0.4208,0.4473,0.4138,0.2435,0.2477,0.0420
ridge,Ridge Classifier,0.4217,0.0000,0.4217,0.4486,0.4127,0.2437,0.2486,0.0420
lr,Logistic Regression,0.4198,0.0000,0.4198,0.4340,0.4113,0.2436,0.2476,0.0680
knn,K Neighbors Classifier,0.4040,0.6812,0.4040,0.4127,0.4012,0.2314,0.2337,0.0560
dt,Decision Tree Classifier,0.3836,0.6172,0.3836,0.3826,0.3813,0.2118,0.2123,0.0640
ada,Ada Boost Classifier,0.3699,0.0000,0.3699,0.3791,0.3665,0.1847,0.1869,0.1340


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Random Forest Classifier,0.4081,0.6986,0.4081,0.4433,0.4111,0.2494,0.2538


=== Results (time order) ===
  checkpoint_upto  n_rows  n_students  \
0            Y1S1     556         556   
1            Y1S2    1326         771   
2            Y2S1    2096         771   
3            Y2S2    2864         771   
4            Y3S1    3628         771   
5            Y3S2    4383         771   
6            Y4S1    5124         771   
7            Y4S2    5877         771   

                                          best_model  test_f1_macro  \
0  GradientBoostingClassifier(ccp_alpha=0.0, crit...       0.286245   
1  ExtraTreesClassifier(bootstrap=False, ccp_alph...       0.303194   
2  RandomForestClassifier(bootstrap=True, ccp_alp...       0.313293   
3  RandomForestClassifier(bootstrap=True, ccp_alp...       0.304486   
4  LogisticRegression(C=1.0, class_weight=None, d...       0.279337   
5  RandomForestClassifier(bootstrap=True, ccp_alp...       0.317350   
6  RandomForestClassifier(bootstrap=True, ccp_alp...       0.328627   
7  RandomForestClassifier(bootstr

In [ ]:
#model name only
results_df["best_model_name"] = results_df["best_model"].astype(str).str.split(r"\(", n=1).str[0]

print(results_df[["checkpoint_upto","best_model_name"]].to_string(index=False))

checkpoint_upto            best_model_name
           Y1S1 GradientBoostingClassifier
           Y1S2       ExtraTreesClassifier
           Y2S1     RandomForestClassifier
           Y2S2     RandomForestClassifier
           Y3S1         LogisticRegression
           Y3S2     RandomForestClassifier
           Y4S1     RandomForestClassifier
           Y4S2     RandomForestClassifier


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import confusion_matrix, classification_report

from pycaret.classification import ClassificationExperiment

DATA_PATH = r"C:\Users\User\Desktop\Final Year Project\Data\FeatureDataset_RiskProgress.xlsx"
SHEETS_ORDER = ["Y1S1","Y1S2","Y2S1","Y2S2","Y3S1","Y3S2","Y4S1","Y4S2"]
TARGET = "RISK_BAND_Y4S2"

CHECKPOINT_UPTO = "Y4S2"  

def load_concat(upto_sheet):
    idx = SHEETS_ORDER.index(upto_sheet) + 1
    frames = []
    for sh in SHEETS_ORDER[:idx]:
        d = pd.read_excel(DATA_PATH, sheet_name=sh)
        d.columns = d.columns.astype(str).str.strip()
        d["REGNO"] = d["REGNO"].astype(str).str.strip()
        frames.append(d)
    out = pd.concat(frames, ignore_index=True)
    out = out[out[TARGET].notna()].copy()
    return out

def group_split(data, test_size=0.2, seed=42):
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    idx = np.arange(len(data))
    tr, te = next(gss.split(idx, groups=data["REGNO"].values))
    return data.iloc[tr].copy(), data.iloc[te].copy()

data = load_concat(CHECKPOINT_UPTO)
train_k, test_k = group_split(data)

exp = ClassificationExperiment()
exp.setup(
    data=train_k.drop(columns=["REGNO"]),
    target=TARGET,
    session_id=42,
    fold=5,
    normalize=True,
    verbose=False
)

best = exp.compare_models(sort="F1")
preds = exp.predict_model(best, data=test_k.drop(columns=["REGNO"]))

y_true = test_k[TARGET].astype(str).values
y_pred = preds["prediction_label"].astype(str).values

# Confusion matrix and classification report
labels_order = ["High Risk","Risk","Moderate","Safe","Very Safe"]
cm = confusion_matrix(y_true, y_pred, labels=labels_order)
cm_df = pd.DataFrame(cm, index=[f"true_{c}" for c in labels_order],
                        columns=[f"pred_{c}" for c in labels_order])

print("Best model:", str(best))
print("\nConfusion Matrix (rows=true, cols=pred):")
print(cm_df)

print("\nClassification Report (precision/recall/f1 per class):")
print(classification_report(y_true, y_pred, labels=labels_order, zero_division=0))

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,0.4338,0.7193,0.4338,0.4399,0.4329,0.2714,0.2726,0.4160
et,Extra Trees Classifier,0.4329,0.7102,0.4329,0.4366,0.4316,0.2710,0.2721,0.2680
lightgbm,Light Gradient Boosting Machine,0.4296,0.7318,0.4296,0.4376,0.4291,0.2650,0.2663,1.2060
gbc,Gradient Boosting Classifier,0.4323,0.0000,0.4323,0.4475,0.4286,0.2634,0.2667,1.9680
lda,Linear Discriminant Analysis,0.4208,0.0000,0.4208,0.4473,0.4138,0.2435,0.2477,0.0500
ridge,Ridge Classifier,0.4217,0.0000,0.4217,0.4486,0.4127,0.2437,0.2486,0.0660
lr,Logistic Regression,0.4198,0.0000,0.4198,0.4340,0.4113,0.2436,0.2476,2.0700
knn,K Neighbors Classifier,0.4040,0.6812,0.4040,0.4127,0.4012,0.2314,0.2337,1.0280
dt,Decision Tree Classifier,0.3836,0.6172,0.3836,0.3826,0.3813,0.2118,0.2123,0.0680
ada,Ada Boost Classifier,0.3699,0.0000,0.3699,0.3791,0.3665,0.1847,0.1869,0.2100


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Random Forest Classifier,0.4081,0.6986,0.4081,0.4433,0.4111,0.2494,0.2538


Best model: RandomForestClassifier(bootstrap=True, ccp_alpha=0.0, class_weight=None,
                       criterion='gini', max_depth=None, max_features='sqrt',
                       max_leaf_nodes=None, max_samples=None,
                       min_impurity_decrease=0.0, min_samples_leaf=1,
                       min_samples_split=2, min_weight_fraction_leaf=0.0,
                       monotonic_cst=None, n_estimators=100, n_jobs=-1,
                       oob_score=False, random_state=42, verbose=0,
                       warm_start=False)

Confusion Matrix (rows=true, cols=pred):
                pred_High Risk  pred_Risk  pred_Moderate  pred_Safe  \
true_High Risk              78         78             41         11   
true_Risk                   21        111             60         25   
true_Moderate               12         93            138         34   
true_Safe                    6         56             97         94   
true_Very Safe              10         21            

In [23]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report

from pycaret.classification import ClassificationExperiment

DATA_PATH = r"C:\Users\User\Desktop\Final Year Project\Data\FeatureDataset_RiskProgress.xlsx"
SHEETS_ORDER = ["Y1S1","Y1S2","Y2S1","Y2S2","Y3S1","Y3S2","Y4S1","Y4S2"]
TARGET = "RISK_BAND_Y4S2"
labels_order = ["High Risk","Risk","Moderate","Safe","Very Safe"]

def load_concat(upto_idx):
    frames = []
    for sh in SHEETS_ORDER[:upto_idx]:
        d = pd.read_excel(DATA_PATH, sheet_name=sh)
        d.columns = d.columns.astype(str).str.strip()
        d["REGNO"] = d["REGNO"].astype(str).str.strip()
        frames.append(d)
    out = pd.concat(frames, ignore_index=True)
    out = out[out[TARGET].notna()].copy()
    return out

def group_split(data, test_size=0.2, seed=42):
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    idx = np.arange(len(data))
    tr, te = next(gss.split(idx, groups=data["REGNO"].values))
    return data.iloc[tr].copy(), data.iloc[te].copy()

rows = []

for k in range(1, len(SHEETS_ORDER)+1):
    data_k = load_concat(k)
    train_k, test_k = group_split(data_k)

    exp = ClassificationExperiment()
    exp.setup(
        data=train_k.drop(columns=["REGNO"]),
        target=TARGET,
        session_id=42,
        fold=5,
        normalize=True,
        verbose=False
    )

    best = exp.compare_models(sort="F1")
    preds = exp.predict_model(best, data=test_k.drop(columns=["REGNO"]))

    y_true = test_k[TARGET].astype(str).values
    y_pred = preds["prediction_label"].astype(str).values

    rep = classification_report(y_true, y_pred, labels=labels_order, output_dict=True, zero_division=0)
    f1_high = rep["High Risk"]["f1-score"]
    recall_high = rep["High Risk"]["recall"]
    precision_high = rep["High Risk"]["precision"]

    rows.append({
        "checkpoint_upto": SHEETS_ORDER[k-1],
        "best_model": str(best).split("(")[0],
        "F1_high_risk": f1_high,
        "Recall_high_risk": recall_high,
        "Precision_high_risk": precision_high,
        "support_high_risk": rep["High Risk"]["support"],
    })

hr_df = pd.DataFrame(rows).sort_values(["F1_high_risk","Recall_high_risk"], ascending=False)

print("=== Ranked by High Risk F1 (then Recall) ===")
print(hr_df.to_string(index=False))

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
gbc,Gradient Boosting Classifier,0.3903,0.0000,0.3903,0.4068,0.3901,0.2149,0.2173,0.4380
rf,Random Forest Classifier,0.3871,0.6716,0.3871,0.3905,0.3805,0.2128,0.2153,0.2400
et,Extra Trees Classifier,0.3839,0.6599,0.3839,0.3959,0.3797,0.2110,0.2139,0.1040
lightgbm,Light Gradient Boosting Machine,0.3484,0.6428,0.3484,0.3529,0.3433,0.1612,0.1633,0.3980
knn,K Neighbors Classifier,0.3548,0.6303,0.3548,0.3524,0.3402,0.1697,0.1745,0.0520
ada,Ada Boost Classifier,0.3290,0.0000,0.3290,0.3314,0.3243,0.1432,0.1443,0.0920
ridge,Ridge Classifier,0.3323,0.0000,0.3323,0.3438,0.3234,0.1416,0.1448,0.0460
lda,Linear Discriminant Analysis,0.3290,0.0000,0.3290,0.3384,0.3224,0.1381,0.1404,0.0360
lr,Logistic Regression,0.3290,0.0000,0.3290,0.3326,0.3220,0.1376,0.1396,0.0640
dt,Decision Tree Classifier,0.3000,0.5525,0.3000,0.3044,0.2984,0.1106,0.1113,0.0900


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Gradient Boosting Classifier,0.2768,0.5617,0.2768,0.2878,0.2695,0.0765,0.0784


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
et,Extra Trees Classifier,0.3576,0.6534,0.3576,0.3600,0.3534,0.1744,0.1759,0.1440
rf,Random Forest Classifier,0.3522,0.6584,0.3522,0.3657,0.3460,0.1624,0.1655,0.1500
gbc,Gradient Boosting Classifier,0.3414,0.0000,0.3414,0.3430,0.3367,0.1524,0.1539,0.6080
lightgbm,Light Gradient Boosting Machine,0.3334,0.6354,0.3334,0.3378,0.3284,0.1409,0.1426,0.7200
lda,Linear Discriminant Analysis,0.3347,0.0000,0.3347,0.3234,0.3189,0.1439,0.1468,0.0720
lr,Logistic Regression,0.3307,0.0000,0.3307,0.3181,0.3171,0.1375,0.1397,0.0460
ridge,Ridge Classifier,0.3361,0.0000,0.3361,0.3118,0.3116,0.1432,0.1474,0.0400
dt,Decision Tree Classifier,0.3117,0.5631,0.3117,0.3114,0.3093,0.1273,0.1277,0.0440
ada,Ada Boost Classifier,0.3117,0.0000,0.3117,0.3098,0.3053,0.1228,0.1239,0.0920
knn,K Neighbors Classifier,0.3064,0.6042,0.3064,0.3057,0.2927,0.1066,0.1093,0.0680


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Extra Trees Classifier,0.3146,0.5845,0.3146,0.3332,0.3007,0.1337,0.1383


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,0.3698,0.6706,0.3698,0.3824,0.3679,0.1855,0.1873,0.2000
gbc,Gradient Boosting Classifier,0.3664,0.0000,0.3664,0.3720,0.3638,0.1844,0.1856,0.9480
lightgbm,Light Gradient Boosting Machine,0.3519,0.6534,0.3519,0.3600,0.3511,0.1642,0.1651,0.9580
lda,Linear Discriminant Analysis,0.3578,0.0000,0.3578,0.3583,0.3503,0.1739,0.1760,0.0440
et,Extra Trees Classifier,0.3484,0.6499,0.3484,0.3526,0.3460,0.1621,0.1632,0.1640
lr,Logistic Regression,0.3518,0.0000,0.3518,0.3520,0.3432,0.1633,0.1655,0.0500
ridge,Ridge Classifier,0.3535,0.0000,0.3535,0.3575,0.3416,0.1652,0.1684,0.0400
knn,K Neighbors Classifier,0.3382,0.6100,0.3382,0.3304,0.3252,0.1480,0.1505,0.0420
ada,Ada Boost Classifier,0.3322,0.0000,0.3322,0.3273,0.3232,0.1450,0.1466,0.1080
dt,Decision Tree Classifier,0.3023,0.5552,0.3023,0.3010,0.3001,0.1125,0.1128,0.0500


...

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,0.3517,0.6432,0.3517,0.3603,0.3518,0.1658,0.1666,0.3160
gbc,Gradient Boosting Classifier,0.3521,0.0000,0.3521,0.3678,0.3485,0.1590,0.1613,1.9260
et,Extra Trees Classifier,0.3472,0.6317,0.3472,0.3522,0.3470,0.1610,0.1616,0.3420
lr,Logistic Regression,0.3510,0.0000,0.3510,0.3713,0.3455,0.1562,0.1587,0.0720
lda,Linear Discriminant Analysis,0.3500,0.0000,0.3500,0.3681,0.3448,0.1541,0.1565,0.0660
ridge,Ridge Classifier,0.3500,0.0000,0.3500,0.3689,0.3400,0.1515,0.1550,0.0440
lightgbm,Light Gradient Boosting Machine,0.3343,0.6454,0.3343,0.3437,0.3346,0.1417,0.1425,1.4040
ada,Ada Boost Classifier,0.3367,0.0000,0.3367,0.3425,0.3313,0.1456,0.1474,0.1380
knn,K Neighbors Classifier,0.3133,0.6008,0.3133,0.3139,0.3067,0.1146,0.1161,0.0720
nb,Naive Bayes,0.3144,0.6233,0.3144,0.3176,0.3033,0.1271,0.1299,0.0460


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Random Forest Classifier,0.3256,0.6236,0.3256,0.3542,0.3270,0.1437,0.1464


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,0.4338,0.7193,0.4338,0.4399,0.4329,0.2714,0.2726,0.3740
et,Extra Trees Classifier,0.4329,0.7102,0.4329,0.4366,0.4316,0.2710,0.2721,0.2880
lightgbm,Light Gradient Boosting Machine,0.4296,0.7318,0.4296,0.4376,0.4291,0.2650,0.2663,1.0360
gbc,Gradient Boosting Classifier,0.4323,0.0000,0.4323,0.4475,0.4286,0.2634,0.2667,1.8940
lda,Linear Discriminant Analysis,0.4208,0.0000,0.4208,0.4473,0.4138,0.2435,0.2477,0.0540
ridge,Ridge Classifier,0.4217,0.0000,0.4217,0.4486,0.4127,0.2437,0.2486,0.0540
lr,Logistic Regression,0.4198,0.0000,0.4198,0.4340,0.4113,0.2436,0.2476,0.1020
knn,K Neighbors Classifier,0.4040,0.6812,0.4040,0.4127,0.4012,0.2314,0.2337,0.1180
dt,Decision Tree Classifier,0.3836,0.6172,0.3836,0.3826,0.3813,0.2118,0.2123,0.0780
ada,Ada Boost Classifier,0.3699,0.0000,0.3699,0.3791,0.3665,0.1847,0.1869,0.1820


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Random Forest Classifier,0.4081,0.6986,0.4081,0.4433,0.4111,0.2494,0.2538


=== Ranked by High Risk F1 (then Recall) ===
checkpoint_upto                 best_model  F1_high_risk  Recall_high_risk  Precision_high_risk  support_high_risk
           Y4S2     RandomForestClassifier      0.453488          0.359447             0.614173              217.0
           Y1S1 GradientBoostingClassifier      0.444444          0.352941             0.600000               17.0
           Y3S2     RandomForestClassifier      0.393701          0.308642             0.543478              162.0
           Y2S2     RandomForestClassifier      0.354430          0.264151             0.538462              106.0
           Y4S1     RandomForestClassifier      0.351351          0.273684             0.490566              190.0
           Y2S1     RandomForestClassifier      0.327586          0.243590             0.500000               78.0
           Y3S1         LogisticRegression      0.290155          0.208955             0.474576              134.0
           Y1S2       ExtraTreesCla

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix

from pycaret.classification import ClassificationExperiment

# Config
DATA_PATH = r"C:\Users\User\Desktop\Final Year Project\Data\FeatureDataset_RiskProgress.xlsx"
SHEETS_ORDER = ["Y1S1","Y1S2","Y2S1","Y2S2","Y3S1","Y3S2","Y4S1","Y4S2"]
TARGET = "RISK_BAND_Y4S2"
LABELS_ORDER = ["High Risk","Risk","Moderate","Safe","Very Safe"]

CHECKPOINTS_TO_TEST = ["Y2S2", "Y4S1"] 
OUT_XLSX = r"C:\Users\User\Desktop\Final Year Project\Data\Contribution_Ablation_Y2S2_Y4S1.xlsx"


def load_concat(upto_sheet: str) -> pd.DataFrame:
    upto_idx = SHEETS_ORDER.index(upto_sheet) + 1
    frames = []
    for sh in SHEETS_ORDER[:upto_idx]:
        d = pd.read_excel(DATA_PATH, sheet_name=sh)
        d.columns = d.columns.astype(str).str.strip()
        d["REGNO"] = d["REGNO"].astype(str).str.strip()
        frames.append(d)
    out = pd.concat(frames, ignore_index=True)
    out = out[out[TARGET].notna()].copy()
    return out

def group_split(data: pd.DataFrame, test_size=0.2, seed=42):
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    idx = np.arange(len(data))
    tr, te = next(gss.split(idx, groups=data["REGNO"].values))
    return data.iloc[tr].copy(), data.iloc[te].copy()

def level_weighted_mark_row(row, weights=(0.2, 0.2, 0.2, 0.4)):
    levels = [1,2,3,4]
    marks = np.array([row.get(f"wavg_mark_L{i}", np.nan) for i in levels], dtype=float)
    creds = np.array([row.get(f"credits_L{i}", 0.0) for i in levels], dtype=float)
    w = np.array(weights, dtype=float)

    ok = (creds > 0) & np.isfinite(marks)
    if ok.sum() == 0:
        return np.nan

    ww = w[ok]
    ww = ww / ww.sum() 
    return float((marks[ok] * ww).sum())

def run_scenario(df_in: pd.DataFrame, feature_cols: list, scenario_name: str):
    use = df_in[["REGNO", TARGET] + feature_cols].copy()
    use = use.dropna(subset=[TARGET])

    train_df, test_df = group_split(use)

    exp = ClassificationExperiment()
    exp.setup(
        data=train_df.drop(columns=["REGNO"]),
        target=TARGET,
        session_id=42,
        fold=5,
        normalize=True,
        verbose=False
    )

    best = exp.compare_models(sort="F1")
    preds = exp.predict_model(best, data=test_df.drop(columns=["REGNO"]))

    y_true = test_df[TARGET].astype(str).values
    y_pred = preds["prediction_label"].astype(str).values

    rep = classification_report(y_true, y_pred, labels=LABELS_ORDER, output_dict=True, zero_division=0)

    cm = confusion_matrix(y_true, y_pred, labels=LABELS_ORDER)
    cm_df = pd.DataFrame(cm,
                         index=[f"true_{c}" for c in LABELS_ORDER],
                         columns=[f"pred_{c}" for c in LABELS_ORDER])

    return {
        "scenario": scenario_name,
        "best_model": str(best).split("(")[0],
        "F1_high_risk": rep["High Risk"]["f1-score"],
        "Recall_high_risk": rep["High Risk"]["recall"],
        "Precision_high_risk": rep["High Risk"]["precision"],
        "Accuracy": rep["accuracy"],
        "support_high_risk": rep["High Risk"]["support"],
        "cm": cm_df
    }

# Feature groupings for ablation
LEVEL_COLS = [
    "credits_L1","credits_L2","credits_L3","credits_L4",
    "wavg_mark_L1","wavg_mark_L2","wavg_mark_L3","wavg_mark_L4",
    "level_weighted_mark_20_20_20_40","L4_share","level_spread_L4_minus_L1"
]

GRADE_COLS = [
    "credits_attempted","module_count",
    "wavg_mark","avg_mark","mark_min","mark_max","mark_std",
    "fail_count","pass_count"
]

TYPE_COLS = [
    "credits_T1","credits_T2","credits_T3","credits_T4","credits_T5",
    "wavg_mark_T1","wavg_mark_T2","wavg_mark_T3","wavg_mark_T4","wavg_mark_T5"
]
RISK_CURR_COL = ["RISK_BAND"]

SCENARIOS = {
    "FULL (Level+Grade+Type+CurrRisk)": LEVEL_COLS + GRADE_COLS + TYPE_COLS + RISK_CURR_COL,

    "NO LEVEL (Grade+Type+CurrRisk)": GRADE_COLS + TYPE_COLS + RISK_CURR_COL,
    "NO GRADE (Level+Type+CurrRisk)": LEVEL_COLS + TYPE_COLS + RISK_CURR_COL,
    "NO TYPE (Level+Grade+CurrRisk)": LEVEL_COLS + GRADE_COLS + RISK_CURR_COL,

    "LEVEL ONLY (+CurrRisk)": LEVEL_COLS + RISK_CURR_COL,
    "GRADE ONLY (+CurrRisk)": GRADE_COLS + RISK_CURR_COL,
    "TYPE ONLY (+CurrRisk)": TYPE_COLS + RISK_CURR_COL,
}


# Run ablation for each checkpoint
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    for ck in CHECKPOINTS_TO_TEST:
        df = load_concat(ck)

        #level weighting
        df["level_weighted_mark_20_20_20_40"] = df.apply(level_weighted_mark_row, axis=1)
        df["L4_share"] = np.where(df["credits_attempted"] > 0, df["credits_L4"] / df["credits_attempted"], np.nan)
        df["level_spread_L4_minus_L1"] = df["wavg_mark_L4"] - df["wavg_mark_L1"]

        outputs = []
        cms = {}

        for name, cols in SCENARIOS.items():
            missing = [c for c in cols if c not in df.columns]
            if missing:
                continue

            out = run_scenario(df, cols, name)
            outputs.append({k: v for k, v in out.items() if k != "cm"})
            cms[name] = out["cm"]

        res = pd.DataFrame(outputs).sort_values(["F1_high_risk","Recall_high_risk"], ascending=False)

        if "FULL (Level+Grade+Type+CurrRisk)" in res["scenario"].values:
            full_row = res[res["scenario"] == "FULL (Level+Grade+Type+CurrRisk)"].iloc[0]
            res["ΔF1_vs_FULL"] = res["F1_high_risk"] - full_row["F1_high_risk"]
            res["ΔRecall_vs_FULL"] = res["Recall_high_risk"] - full_row["Recall_high_risk"]
        else:
            res["ΔF1_vs_FULL"] = np.nan
            res["ΔRecall_vs_FULL"] = np.nan

        sheet_res = f"{ck}_ablation"
        res.to_excel(writer, sheet_name=sheet_res, index=False)

        best_scenario = res.iloc[0]["scenario"]
        cm_df = cms[best_scenario]
        sheet_cm = f"{ck}_CM_best"
        cm_df.to_excel(writer, sheet_name=sheet_cm)

print("Saved:", OUT_XLSX)

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,0.3689,0.6625,0.3689,0.3755,0.3655,0.1848,0.1865,0.2960
lda,Linear Discriminant Analysis,0.3508,0.0000,0.3508,0.3561,0.3475,0.1634,0.1647,0.0320
gbc,Gradient Boosting Classifier,0.3495,0.0000,0.3495,0.3551,0.3469,0.1613,0.1625,1.1020
lightgbm,Light Gradient Boosting Machine,0.3464,0.6509,0.3464,0.3548,0.3452,0.1570,0.1581,0.9980
et,Extra Trees Classifier,0.3464,0.6560,0.3464,0.3508,0.3445,0.1577,0.1587,0.1520
lr,Logistic Regression,0.3446,0.0000,0.3446,0.3476,0.3393,0.1547,0.1561,0.0760
ridge,Ridge Classifier,0.3421,0.0000,0.3421,0.3515,0.3356,0.1481,0.1503,0.0600
ada,Ada Boost Classifier,0.3134,0.0000,0.3134,0.3108,0.3093,0.1231,0.1237,0.1120
knn,K Neighbors Classifier,0.3140,0.6096,0.3140,0.3088,0.3022,0.1141,0.1159,0.0620
dt,Decision Tree Classifier,0.2859,0.5420,0.2859,0.2835,0.2836,0.0889,0.0891,0.0720


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Random Forest Classifier,0.3078,0.6215,0.3078,0.3375,0.3076,0.1245,0.1273


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,0.3602,0.6502,0.3602,0.3634,0.3568,0.1748,0.1760,0.2080
lightgbm,Light Gradient Boosting Machine,0.3495,0.6398,0.3495,0.3552,0.3489,0.1625,0.1632,0.8860
lr,Logistic Regression,0.3564,0.0000,0.3564,0.3586,0.3474,0.1692,0.1714,0.0420
lda,Linear Discriminant Analysis,0.3545,0.0000,0.3545,0.3524,0.3457,0.1678,0.1697,0.0360
ridge,Ridge Classifier,0.3533,0.0000,0.3533,0.3548,0.3403,0.1624,0.1655,0.0300
gbc,Gradient Boosting Classifier,0.3402,0.0000,0.3402,0.3500,0.3393,0.1482,0.1494,0.9080
knn,K Neighbors Classifier,0.3395,0.6147,0.3395,0.3388,0.3330,0.1509,0.1524,0.0400
ada,Ada Boost Classifier,0.3327,0.0000,0.3327,0.3290,0.3272,0.1456,0.1463,0.1020
et,Extra Trees Classifier,0.3283,0.6452,0.3283,0.3303,0.3266,0.1362,0.1369,0.1620
dt,Decision Tree Classifier,0.3184,0.5639,0.3184,0.3189,0.3177,0.1313,0.1315,0.0360


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Random Forest Classifier,0.3304,0.6317,0.3304,0.3666,0.3296,0.1533,0.1572


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
et,Extra Trees Classifier,0.3621,0.6524,0.3621,0.3672,0.3603,0.1788,0.1799,0.1520
rf,Random Forest Classifier,0.3595,0.6621,0.3595,0.3741,0.3570,0.1705,0.1725,0.2000
lightgbm,Light Gradient Boosting Machine,0.3452,0.6510,0.3452,0.3499,0.3429,0.1559,0.1567,0.8260
gbc,Gradient Boosting Classifier,0.3396,0.0000,0.3396,0.3489,0.3382,0.1483,0.1492,0.7420
lda,Linear Discriminant Analysis,0.3402,0.0000,0.3402,0.3484,0.3349,0.1476,0.1491,0.0320
ridge,Ridge Classifier,0.3421,0.0000,0.3421,0.3582,0.3333,0.1462,0.1487,0.0420
lr,Logistic Regression,0.3390,0.0000,0.3390,0.3466,0.3319,0.1446,0.1463,0.0460
knn,K Neighbors Classifier,0.3383,0.6204,0.3383,0.3441,0.3311,0.1451,0.1473,0.0400
ada,Ada Boost Classifier,0.3234,0.0000,0.3234,0.3238,0.3214,0.1342,0.1347,0.0920
nb,Naive Bayes,0.3240,0.6346,0.3240,0.3227,0.2917,0.1335,0.1497,0.0380


...

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
gbc,Gradient Boosting Classifier,0.3486,0.0000,0.3486,0.3577,0.3450,0.1573,0.1589,0.8420
ada,Ada Boost Classifier,0.3378,0.0000,0.3378,0.3460,0.3311,0.1416,0.1438,0.0980
rf,Random Forest Classifier,0.3339,0.6226,0.3339,0.3358,0.3308,0.1420,0.1429,0.2340
lightgbm,Light Gradient Boosting Machine,0.3283,0.6275,0.3283,0.3318,0.3264,0.1352,0.1359,0.8220
et,Extra Trees Classifier,0.3231,0.6143,0.3231,0.3208,0.3197,0.1303,0.1308,0.1640
ridge,Ridge Classifier,0.3290,0.0000,0.3290,0.3551,0.3116,0.1199,0.1252,0.0260
knn,K Neighbors Classifier,0.3137,0.5931,0.3137,0.3174,0.3083,0.1175,0.1187,0.0500
lda,Linear Discriminant Analysis,0.3175,0.0000,0.3175,0.3376,0.3055,0.1069,0.1108,0.0320
lr,Logistic Regression,0.3242,0.0000,0.3242,0.3430,0.3047,0.1154,0.1201,0.0380
dt,Decision Tree Classifier,0.2920,0.5530,0.2920,0.2883,0.2885,0.0927,0.0930,0.0420


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Gradient Boosting Classifier,0.2954,0.6207,0.2954,0.3222,0.2902,0.1085,0.1119


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lda,Linear Discriminant Analysis,0.3343,0.0000,0.3343,0.3563,0.3163,0.1277,0.1342,0.0380
lr,Logistic Regression,0.3388,0.0000,0.3388,0.3559,0.3137,0.1337,0.1416,0.0680
ada,Ada Boost Classifier,0.3214,0.0000,0.3214,0.3264,0.3108,0.1233,0.1271,0.0920
gbc,Gradient Boosting Classifier,0.3214,0.0000,0.3214,0.3390,0.3108,0.1156,0.1207,0.6400
ridge,Ridge Classifier,0.3367,0.0000,0.3367,0.3576,0.3100,0.1285,0.1371,0.0380
lightgbm,Light Gradient Boosting Machine,0.3137,0.6158,0.3137,0.3269,0.3054,0.1082,0.1122,0.8140
qda,Quadratic Discriminant Analysis,0.3154,0.0000,0.3154,0.3412,0.3007,0.1259,0.1356,0.0380
rf,Random Forest Classifier,0.3109,0.6105,0.3109,0.3173,0.2988,0.1048,0.1089,0.1980
et,Extra Trees Classifier,0.3081,0.6040,0.3081,0.3153,0.2979,0.1023,0.1060,0.1660
dt,Decision Tree Classifier,0.2844,0.5456,0.2844,0.2865,0.2753,0.0754,0.0775,0.0420


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Linear Discriminant Analysis,0.2993,0.6094,0.2993,0.3667,0.2875,0.1185,0.1307


Saved: C:\Users\User\Desktop\Final Year Project\Data\Contribution_Ablation_Y2S2_Y4S1.xlsx
